# Melrose Refinery Problem with AMPL in Python
## AMPL Modeling for Problem 3

## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

```python
%pip install amplpy
```

In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [2]:
%pip install amplpy
from amplpy import AMPL, modules

modules.install('highs')


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl

## Problem: Melrose Refinery Profit Maximization

The refinery produces gasoline and fuel oil from crude oil.

**Products:**
- Gasoline: $8/barrel, minimum average grade 9, max 2,000 barrels
- Fuel oil: $6/barrel, minimum average grade 7, max 600 barrels

**Processing methods (yield per barrel of crude):**

| Method | Grade 6 | Grade 8 | Grade 10 | Cost |
|--------|---------|---------|----------|------|
| 1      | 0.2     | 0.2     | 0.6      | $3.40|
| 2      | 0.3     | 0.3     | 0.4      | $3.00|
| 3      | 0.4     | 0.4     | 0.2      | $2.60|

**Catalytic cracking:**
- Grade 6 → Grade 8: $1.30/barrel
- Grade 8 → Grade 10: $2.00/barrel

**Excess disposal:** $0.20/barrel

Maximize refinery profit.

In [4]:
@timed
def solve_refinery(solver: str = "highs"):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        var crude {1..3} >= 0;

        var g6_to_gas >= 0;
        var g8_to_gas >= 0;
        var g10_to_gas >= 0;

        var g6_to_fuel >= 0;
        var g8_to_fuel >= 0;
        var g10_to_fuel >= 0;

        var crack_6_to_8 >= 0;
        var crack_8_to_10 >= 0;

        var dispose6 >= 0;
        var dispose8 >= 0;
        var dispose10 >= 0;

        maximize Profit:
            8 * (g6_to_gas + g8_to_gas + g10_to_gas)
            + 6 * (g6_to_fuel + g8_to_fuel + g10_to_fuel)
            - 3.4 * crude[1] - 3.0 * crude[2] - 2.6 * crude[3]
            - 1.30 * crack_6_to_8
            - 2.00 * crack_8_to_10
            - 0.20 * (dispose6 + dispose8 + dispose10);

        subject to BalanceG6:
            0.2*crude[1] + 0.3*crude[2] + 0.4*crude[3]
            = g6_to_gas + g6_to_fuel + crack_6_to_8 + dispose6;

        subject to BalanceG8:
            0.2*crude[1] + 0.3*crude[2] + 0.4*crude[3] + crack_6_to_8
            = g8_to_gas + g8_to_fuel + crack_8_to_10 + dispose8;

        subject to BalanceG10:
            0.6*crude[1] + 0.4*crude[2] + 0.2*crude[3] + crack_8_to_10
            = g10_to_gas + g10_to_fuel + dispose10;

        subject to GasolineQuality:
            6*g6_to_gas + 8*g8_to_gas + 10*g10_to_gas
            >= 9 * (g6_to_gas + g8_to_gas + g10_to_gas);

        subject to FuelOilQuality:
            6*g6_to_fuel + 8*g8_to_fuel + 10*g10_to_fuel
            >= 7 * (g6_to_fuel + g8_to_fuel + g10_to_fuel);

        subject to GasolineLimit:
            g6_to_gas + g8_to_gas + g10_to_gas <= 2000;

        subject to FuelOilLimit:
            g6_to_fuel + g8_to_fuel + g10_to_fuel <= 600;
        '''
    )
    ampl.solve(solver=solver)

    var_names = ["g6_to_gas", "g8_to_gas", "g10_to_gas",
                 "g6_to_fuel", "g8_to_fuel", "g10_to_fuel",
                 "crack_6_to_8", "crack_8_to_10",
                 "dispose6", "dispose8", "dispose10"]
    solution = {v: round(ampl.get_value(v), 4) for v in var_names}

    crude_vals = ampl.get_variable("crude").get_values().to_dict()
    for k, v in crude_vals.items():
        key = k if not isinstance(k, tuple) else k[0]
        solution[f"crude_{key}"] = round(v, 4)

    solution["profit"] = round(ampl.get_value("Profit"), 4)
    return solution

In [5]:
result, elapsed = solve_refinery()

print("=== MELROSE REFINERY RESULTS ===")
print(f"\nCrude processed:")
for m in [1, 2, 3]:
    key = f"crude_{m}"
    if key in result:
        print(f"  Method {m}: {result[key]:.2f} barrels")

gas_total = result["g6_to_gas"] + result["g8_to_gas"] + result["g10_to_gas"]
fuel_total = result["g6_to_fuel"] + result["g8_to_fuel"] + result["g10_to_fuel"]

print(f"\nGasoline production: {gas_total:.2f} barrels")
print(f"  Grade 6: {result['g6_to_gas']:.2f}, Grade 8: {result['g8_to_gas']:.2f}, Grade 10: {result['g10_to_gas']:.2f}")
if gas_total > 0:
    avg_gas = (6*result['g6_to_gas'] + 8*result['g8_to_gas'] + 10*result['g10_to_gas']) / gas_total
    print(f"  Average grade: {avg_gas:.2f} (need >= 9)")

print(f"\nFuel oil production: {fuel_total:.2f} barrels")
print(f"  Grade 6: {result['g6_to_fuel']:.2f}, Grade 8: {result['g8_to_fuel']:.2f}, Grade 10: {result['g10_to_fuel']:.2f}")
if fuel_total > 0:
    avg_fuel = (6*result['g6_to_fuel'] + 8*result['g8_to_fuel'] + 10*result['g10_to_fuel']) / fuel_total
    print(f"  Average grade: {avg_fuel:.2f} (need >= 7)")

print(f"\nCracking:")
print(f"  Grade 6 -> 8: {result['crack_6_to_8']:.2f} barrels")
print(f"  Grade 8 -> 10: {result['crack_8_to_10']:.2f} barrels")

dispose_total = result["dispose6"] + result["dispose8"] + result["dispose10"]
print(f"\nDisposed: {dispose_total:.2f} barrels")
print(f"\nMaximum profit: ${result['profit']:.2f}")
print(f"Time: {elapsed:.8f} seconds")

HiGHS 1.11.0=== MELROSE REFINERY RESULTS ===

Crude processed:
  Method 1: 0.00 barrels
  Method 2: 2400.00 barrels
  Method 3: 200.00 barrels

Gasoline production: 2000.00 barrels
  Grade 6: 0.00, Grade 8: 1000.00, Grade 10: 1000.00
  Average grade: 9.00 (need >= 9)

Fuel oil production: 600.00 barrels
  Grade 6: 300.00, Grade 8: 300.00, Grade 10: 0.00
  Average grade: 7.00 (need >= 7)

Cracking:
  Grade 6 -> 8: 500.00 barrels
  Grade 8 -> 10: 0.00 barrels

Disposed: 0.00 barrels

Maximum profit: $11230.00
Time: 0.01159826 seconds


## Sensitivity Analysis

In [6]:
ampl = create_ampl_instance()

ampl.eval(
    r'''
    var crude {1..3} >= 0;
    var g6_to_gas >= 0;
    var g8_to_gas >= 0;
    var g10_to_gas >= 0;
    var g6_to_fuel >= 0;
    var g8_to_fuel >= 0;
    var g10_to_fuel >= 0;
    var crack_6_to_8 >= 0;
    var crack_8_to_10 >= 0;
    var dispose6 >= 0;
    var dispose8 >= 0;
    var dispose10 >= 0;

    maximize Profit:
        8 * (g6_to_gas + g8_to_gas + g10_to_gas)
        + 6 * (g6_to_fuel + g8_to_fuel + g10_to_fuel)
        - 3.4 * crude[1] - 3.0 * crude[2] - 2.6 * crude[3]
        - 1.30 * crack_6_to_8
        - 2.00 * crack_8_to_10
        - 0.20 * (dispose6 + dispose8 + dispose10);

    subject to BalanceG6:
        0.2*crude[1] + 0.3*crude[2] + 0.4*crude[3]
        = g6_to_gas + g6_to_fuel + crack_6_to_8 + dispose6;

    subject to BalanceG8:
        0.2*crude[1] + 0.3*crude[2] + 0.4*crude[3] + crack_6_to_8
        = g8_to_gas + g8_to_fuel + crack_8_to_10 + dispose8;

    subject to BalanceG10:
        0.6*crude[1] + 0.4*crude[2] + 0.2*crude[3] + crack_8_to_10
        = g10_to_gas + g10_to_fuel + dispose10;

    subject to GasolineQuality:
        6*g6_to_gas + 8*g8_to_gas + 10*g10_to_gas
        >= 9 * (g6_to_gas + g8_to_gas + g10_to_gas);

    subject to FuelOilQuality:
        6*g6_to_fuel + 8*g8_to_fuel + 10*g10_to_fuel
        >= 7 * (g6_to_fuel + g8_to_fuel + g10_to_fuel);

    subject to GasolineLimit:
        g6_to_gas + g8_to_gas + g10_to_gas <= 2000;

    subject to FuelOilLimit:
        g6_to_fuel + g8_to_fuel + g10_to_fuel <= 600;
    '''
)
ampl.solve()

constraints = ["BalanceG6", "BalanceG8", "BalanceG10",
               "GasolineQuality", "FuelOilQuality",
               "GasolineLimit", "FuelOilLimit"]

print("=== SHADOW PRICES AND SLACK ===")
print(f"{'Constraint':<20} {'Shadow Price':>14} {'Slack':>10}")
print("-" * 46)
for c in constraints:
    con = ampl.get_constraint(c)
    print(f"{c:<20} {con.dual():>14.4f} {con.slack():>10.4f}")

print()
print("=== KEY INSIGHTS ===")
gas_limit_dual = ampl.get_constraint("GasolineLimit").dual()
fuel_limit_dual = ampl.get_constraint("FuelOilLimit").dual()
print(f"Value of one more barrel of gasoline capacity: ${gas_limit_dual:.4f}")
print(f"Value of one more barrel of fuel oil capacity: ${fuel_limit_dual:.4f}")

HiGHS 1.11.0=== SHADOW PRICES AND SLACK ===
Constraint             Shadow Price      Slack
----------------------------------------------
BalanceG6                   -1.5500     0.0000
BalanceG8                   -2.8500     0.0000
BalanceG10                  -4.2000     0.0000
GasolineQuality             -0.6750     0.0000
FuelOilQuality              -0.6500     0.0000
GasolineLimit                4.4750     0.0000
FuelOilLimit                 3.8000     0.0000

=== KEY INSIGHTS ===
Value of one more barrel of gasoline capacity: $4.4750
Value of one more barrel of fuel oil capacity: $3.8000


## Interpretation

The sensitivity analysis for the refinery problem reveals:

- **Balance constraints** (G6, G8, G10): Their shadow prices reflect the marginal value of each grade of processed crude. Higher-grade crude is more valuable because it requires less cracking.
- **Quality constraints**: If binding, the shadow price shows the cost of maintaining quality standards. Relaxing the gasoline grade from 9 to 8.9 would allow cheaper crude mixes.
- **Demand limits**: The shadow price on gasoline/fuel oil limits shows the marginal profit from selling one more barrel.
- **Cracking decisions**: The solution reveals whether catalytic cracking is economically justified — converting grade 6 to 8 costs $1.30 and grade 8 to 10 costs $2.00.